In [ ]:
from copy import deepcopy
import json
import numpy as np

In [ ]:
BENCHMARK = 'triviaqa'
EXP_NAME = 'trivia_qwen257b_medium_mixture(#f8309682)(v2.3.7)'

LOAD_LLMCALLS_STATISTICS_PATH = f"./artifacts/{BENCHMARK}/{EXP_NAME}/llmcalls_statistics.json"
LOAD_LLMTASK_MEAN_TOKENS_PER_BENCHMARK_PATH = './artifacts/pai2_llmtask_mean_tokens_per_benchmark.json'
LOAD_LLMCALL_COUNTERS_PATH = f"artifacts/{BENCHMARK}/{EXP_NAME}/llmcalls_counters.json"

SAVE_LLMTOKENS_STATISTICS_PATH = f"./artifacts/{BENCHMARK}/{EXP_NAME}/llmtokens_statistics.json"

In [ ]:
with open(LOAD_LLMTASK_MEAN_TOKENS_PER_BENCHMARK_PATH, 'r', encoding='utf-8') as fd:
    LLMTASK_MEAN_TOKENS_PER_BENCHMARK = json.loads(fd.read())

with open(LOAD_LLMCALL_COUNTERS_PATH, 'r', encoding='utf-8') as fd:
    LLMCALL_COUNTERS = json.loads(fd.read())

with open(LOAD_LLMCALLS_STATISTICS_PATH, 'r', encoding='utf-8') as fd:
    LLMCALLS_STATISTICS = json.loads(fd.read())

In [ ]:
LLMTOKENS_STATISTICS  = {
    'summary': None,

    'answer_classify_solver': None,
    'casual_answer_gen_solver': None,
    'strict_answer_gen_solver': None,

    'clueanswers_summarizer_solver': None,
    'clueanswer_generator_solver': None,
    'cluequeries_generator_solver': None,
    'entities_extractor_solver': None,

    'searchplan_enhance_classify_solver': None,
    'plan_enhancing_solver': None,
    'plan_initialing_solver': None #
}

LLM_TASKS = set(LLMTOKENS_STATISTICS.keys()).difference(['summary'])
STATISTICS = ['min', 'mean', 'median', 'max', 'std', 'sum']

In [ ]:
for llmtask_name in LLM_TASKS:
    LLMTOKENS_STATISTICS[llmtask_name] = {
        stat_name: {
            'generated': round(LLMCALLS_STATISTICS[llmtask_name][stat_name] * LLMTASK_MEAN_TOKENS_PER_BENCHMARK[BENCHMARK][llmtask_name]['generated'],3),
            'prompt': round(LLMCALLS_STATISTICS[llmtask_name][stat_name] * LLMTASK_MEAN_TOKENS_PER_BENCHMARK[BENCHMARK][llmtask_name]['prompt'],3)
        }
        for stat_name in STATISTICS
    }
    LLMTOKENS_STATISTICS[llmtask_name]['count'] = LLMCALLS_STATISTICS[llmtask_name]['count']

generated_llmtokens_per_query = []
prompt_llmtokens_per_query = []
for query_idx, llmcall_counters in LLMCALL_COUNTERS.items():
    generated_llmtokens_per_query.append(sum([
        llmcalls_count * LLMTASK_MEAN_TOKENS_PER_BENCHMARK[BENCHMARK][llmtask_name]['generated'] 
        for llmtask_name, llmcalls_count in llmcall_counters.items()
    ]))
    prompt_llmtokens_per_query.append(sum([
        llmcalls_count * LLMTASK_MEAN_TOKENS_PER_BENCHMARK[BENCHMARK][llmtask_name]['prompt'] 
        for llmtask_name, llmcalls_count in llmcall_counters.items()
    ]))

LLMTOKENS_STATISTICS['summary'] = {
    'count': {
        'generated': len(generated_llmtokens_per_query),
        'prompt': len(prompt_llmtokens_per_query)
    },
    'max': {
        'generated': max(generated_llmtokens_per_query),
        'prompt': max(prompt_llmtokens_per_query)
    },
    'mean': {
        'generated': round(float(np.mean(generated_llmtokens_per_query)),3),
        'prompt': round(float(np.mean(prompt_llmtokens_per_query)), 3)
    },
    'median': {
        'generated': float(np.median(generated_llmtokens_per_query)),
        'prompt': float(np.median(prompt_llmtokens_per_query))
    },
    'min': {
        'generated': min(generated_llmtokens_per_query),
        'prompt': min(prompt_llmtokens_per_query)
    },
    'std': {
        'generated': round(float(np.std(generated_llmtokens_per_query)), 3),
        'prompt': round(float(np.std(prompt_llmtokens_per_query)), 3)
    },
    'sum': {
        'generated': sum(generated_llmtokens_per_query),
        'prompt': sum(prompt_llmtokens_per_query)
    },
}

In [ ]:
LLMTOKENS_STATISTICS

In [ ]:
with open(SAVE_LLMTOKENS_STATISTICS_PATH, 'w', encoding='utf-8') as fd:
    fd.write(json.dumps(LLMTOKENS_STATISTICS, ensure_ascii=False, indent=3))